# 03 - Train a U-Net

**Goal:** Train a segmentation model end-to-end on synthetic data.

---

## Setup

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## Step 1: Create Synthetic Dataset

We'll generate images with random shapes (circles, squares) and their segmentation masks.

In [ ]:
class ShapesDataset(Dataset):
    """Dataset of synthetic shapes for segmentation."""
    
    def __init__(self, num_samples=500, image_size=128):
        self.num_samples = num_samples
        self.image_size = image_size
        # Pre-generate data for consistency
        self.data = [self._generate_sample() for _ in range(num_samples)]
    
    def _generate_sample(self):
        size = self.image_size
        image = np.random.normal(0.2, 0.05, (size, size)).astype(np.float32)
        mask = np.zeros((size, size), dtype=np.int64)
        
        # Add 1-3 random circles (class 1)
        num_circles = np.random.randint(1, 4)
        for _ in range(num_circles):
            cx = np.random.randint(20, size - 20)
            cy = np.random.randint(20, size - 20)
            r = np.random.randint(8, 20)
            y, x = np.ogrid[:size, :size]
            circle = np.sqrt((x - cx)**2 + (y - cy)**2) <= r
            image[circle] = np.clip(image[circle] + 0.6, 0, 1)
            mask[circle] = 1
        
        # Add 0-2 random squares (class 2)
        num_squares = np.random.randint(0, 3)
        for _ in range(num_squares):
            sx = np.random.randint(10, size - 30)
            sy = np.random.randint(10, size - 30)
            side = np.random.randint(10, 25)
            image[sy:sy+side, sx:sx+side] = np.clip(
                image[sy:sy+side, sx:sx+side] + 0.4, 0, 1
            )
            mask[sy:sy+side, sx:sx+side] = 2
        
        return image, mask
    
    def __len__(self):
        return self.num_samples
    
    def __getitem__(self, idx):
        image, mask = self.data[idx]
        # Add channel dimension and convert to tensor
        image = torch.tensor(image).unsqueeze(0)  # (1, H, W)
        mask = torch.tensor(mask)  # (H, W)
        return image, mask

# Create datasets
train_dataset = ShapesDataset(num_samples=400)
val_dataset = ShapesDataset(num_samples=100)

print(f"Training samples: {len(train_dataset)}")
print(f"Validation samples: {len(val_dataset)}")

In [ ]:
# Visualize samples
fig, axes = plt.subplots(2, 4, figsize=(14, 7))

for i in range(4):
    image, mask = train_dataset[i]
    
    axes[0, i].imshow(image.squeeze(), cmap='gray')
    axes[0, i].set_title(f'Image {i}')
    axes[0, i].axis('off')
    
    axes[1, i].imshow(mask, cmap='tab10', vmin=0, vmax=3)
    axes[1, i].set_title(f'Mask {i}\n(0=bg, 1=circle, 2=square)')
    axes[1, i].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# Create data loaders
batch_size = 8

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

# Check batch shapes
images, masks = next(iter(train_loader))
print(f"Image batch shape: {images.shape}")
print(f"Mask batch shape: {masks.shape}")

## Step 2: Define U-Net Model

Using the architecture from the previous notebook:

In [ ]:
class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.double_conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )
    
    def forward(self, x):
        return self.double_conv(x)


class UNet(nn.Module):
    def __init__(self, in_channels=1, out_channels=3, features=[32, 64, 128, 256]):
        super().__init__()
        
        # Encoder
        self.encoder = nn.ModuleList()
        self.pool = nn.MaxPool2d(2)
        
        prev_channels = in_channels
        for feature in features:
            self.encoder.append(DoubleConv(prev_channels, feature))
            prev_channels = feature
        
        # Bottleneck
        self.bottleneck = DoubleConv(features[-1], features[-1] * 2)
        
        # Decoder
        self.upconvs = nn.ModuleList()
        self.decoder = nn.ModuleList()
        
        reversed_features = features[::-1]
        prev_channels = features[-1] * 2
        
        for feature in reversed_features:
            self.upconvs.append(
                nn.ConvTranspose2d(prev_channels, feature, kernel_size=2, stride=2)
            )
            self.decoder.append(DoubleConv(feature * 2, feature))
            prev_channels = feature
        
        # Final conv
        self.final_conv = nn.Conv2d(features[0], out_channels, kernel_size=1)
    
    def forward(self, x):
        # Encoder with skip connections
        skip_connections = []
        
        for encoder_block in self.encoder:
            x = encoder_block(x)
            skip_connections.append(x)
            x = self.pool(x)
        
        x = self.bottleneck(x)
        
        # Decoder
        skip_connections = skip_connections[::-1]
        
        for i, (upconv, decoder_block) in enumerate(zip(self.upconvs, self.decoder)):
            x = upconv(x)
            skip = skip_connections[i]
            x = torch.cat([skip, x], dim=1)
            x = decoder_block(x)
        
        return self.final_conv(x)


# Create model
model = UNet(in_channels=1, out_channels=3).to(device)  # 3 classes: bg, circle, square
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

## Step 3: Loss Function and Optimizer

In [ ]:
# Loss: Cross-Entropy for multi-class segmentation
criterion = nn.CrossEntropyLoss()

# Optimizer
optimizer = optim.Adam(model.parameters(), lr=1e-3)

# Metrics
def dice_score(pred, target, num_classes=3):
    """Calculate mean Dice score across classes."""
    pred = pred.argmax(dim=1)  # (B, H, W)
    dice_scores = []
    
    for c in range(1, num_classes):  # Skip background
        pred_c = (pred == c).float()
        target_c = (target == c).float()
        
        intersection = (pred_c * target_c).sum()
        union = pred_c.sum() + target_c.sum()
        
        if union > 0:
            dice_scores.append((2 * intersection / union).item())
    
    return np.mean(dice_scores) if dice_scores else 0.0

## Step 4: Training Loop

In [ ]:
def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0
    total_dice = 0
    
    for images, masks in loader:
        images = images.to(device)
        masks = masks.to(device)
        
        # Forward
        outputs = model(images)
        loss = criterion(outputs, masks)
        
        # Backward
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        total_dice += dice_score(outputs, masks)
    
    return total_loss / len(loader), total_dice / len(loader)


def validate(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    total_dice = 0
    
    with torch.no_grad():
        for images, masks in loader:
            images = images.to(device)
            masks = masks.to(device)
            
            outputs = model(images)
            loss = criterion(outputs, masks)
            
            total_loss += loss.item()
            total_dice += dice_score(outputs, masks)
    
    return total_loss / len(loader), total_dice / len(loader)

In [ ]:
# Training!
num_epochs = 20

history = {
    'train_loss': [], 'val_loss': [],
    'train_dice': [], 'val_dice': []
}

print("Starting training...\n")

for epoch in range(num_epochs):
    train_loss, train_dice = train_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_dice = validate(model, val_loader, criterion, device)
    
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['train_dice'].append(train_dice)
    history['val_dice'].append(val_dice)
    
    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f"Epoch {epoch+1:2d}/{num_epochs}:")
        print(f"  Train Loss: {train_loss:.4f}, Train Dice: {train_dice:.4f}")
        print(f"  Val Loss:   {val_loss:.4f}, Val Dice:   {val_dice:.4f}")

print("\nTraining complete!")

In [ ]:
# Plot training curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(history['train_loss'], label='Train')
ax1.plot(history['val_loss'], label='Validation')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Loss')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(history['train_dice'], label='Train')
ax2.plot(history['val_dice'], label='Validation')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Dice Score')
ax2.set_title('Dice Score (higher is better)')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Step 5: Visualize Predictions

In [ ]:
# Get some validation samples
model.eval()

fig, axes = plt.subplots(4, 3, figsize=(12, 16))

with torch.no_grad():
    for i in range(4):
        image, mask = val_dataset[i * 10]
        
        # Predict
        pred = model(image.unsqueeze(0).to(device))
        pred = pred.argmax(dim=1).squeeze().cpu().numpy()
        
        # Plot
        axes[i, 0].imshow(image.squeeze(), cmap='gray')
        axes[i, 0].set_title('Input')
        axes[i, 0].axis('off')
        
        axes[i, 1].imshow(mask, cmap='tab10', vmin=0, vmax=3)
        axes[i, 1].set_title('Ground Truth')
        axes[i, 1].axis('off')
        
        axes[i, 2].imshow(pred, cmap='tab10', vmin=0, vmax=3)
        axes[i, 2].set_title('Prediction')
        axes[i, 2].axis('off')

plt.suptitle('Segmentation Results\n(0=background, 1=circle, 2=square)', fontsize=14)
plt.tight_layout()
plt.show()

## Step 6: Save the Model

In [ ]:
# Save model
torch.save(model.state_dict(), '/workspace/data/unet_shapes.pth')
print("Model saved!")

# How to load:
# model = UNet(in_channels=1, out_channels=3)
# model.load_state_dict(torch.load('/workspace/data/unet_shapes.pth'))
# model.eval()

## What We Did

1. **Created synthetic dataset** - Random shapes with masks
2. **Built U-Net** - Encoder-decoder with skip connections
3. **Trained** - CrossEntropyLoss + Adam optimizer
4. **Evaluated** - Dice score as metric
5. **Visualized** - Predictions vs ground truth

This is the same workflow for real medical imaging - just swap the data!

**Next:** MONAI introduction - the framework your production code uses.